In [0]:
%run ./00_setup_config

In [0]:
# ==========================================================
#  PARAMETROS DA BRONZE
# ==========================================================
CATALOGO        = "workspace"
ESQUEMA         = SQUAD
TABELA_BRONZE   = f"{CATALOGO}.{ESQUEMA}.bronze_{TABELA}"

INTERVALO_S     = 5      # polling definido para o projeto
DURACAO_MAX_MIN = 30     # teto: o loop nunca passa disso
OCIOSO_MAX_MIN  = 10     # para antes se ficar esse tempo sem arquivo novo
LOOP_ATIVO      = True   # False = uma passada so

# O carimbo dos diretorios do lake e as colunas dt_* estao no horario de
# Brasilia; a particao segue o mesmo relogio para bater com eles.
FUSO_NEGOCIO     = "America/Sao_Paulo"
COLUNAS_PARTICAO = ["ano", "mes", "dia", "hora"]
# ==========================================================

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{ESQUEMA}")

print(f"Origem : {CONTAINER}/{PREFIXO}/{PADRAO_LOTES}/{TABELA}.parquet")
print(f"Bronze : {TABELA_BRONZE}  (append, particao {'/'.join(COLUNAS_PARTICAO)})")
modo = f"loop a cada {INTERVALO_S}s, max {DURACAO_MAX_MIN} min, para apos {OCIOSO_MAX_MIN} min ocioso"
print(f"Modo   : {modo if LOOP_ATIVO else 'uma passada'}")

In [0]:
import time
from datetime import datetime
from zoneinfo import ZoneInfo

from pyspark.sql import functions as F

RAIZ_CONTAINER = f"abfss://{CONTAINER}@{_sufixo}/"


def _relativo(coluna):
    """abfss://raw@<conta>.dfs.core.windows.net/real-time-data/... -> real-time-data/..."""
    return F.regexp_extract(coluna, rf"({PREFIXO}/.+)$", 1)


def _traduzir_erro(e):
    msg = str(e)
    if "AuthorizationPermissionMismatch" in msg or "403" in msg:
        return PermissionError("403: o service principal autenticou, mas nao tem leitura no container.")
    if "AADSTS" in msg or "invalid_client" in msg or "401" in msg:
        return PermissionError("401: credencial recusada - conferir ADLS_CLIENT_ID/SECRET/TENANT_ID no .env.")
    return None


def listar_arquivos():
    """Arquivos da tabela presentes agora no lake, como caminhos relativos."""
    try:
        linhas = (
            spark.read.format("binaryFile").options(**adls_options)
            .load(CAMINHO_ORIGEM)
            .select(_relativo(F.col("path")).alias("arquivo"))
            .collect()
        )
    except Exception as e:
        if "PATH_NOT_FOUND" in str(e) or "does not exist" in str(e):
            return set()  # nenhuma janela de tempo real aberta ainda
        traduzido = _traduzir_erro(e)
        if traduzido:
            raise traduzido from None
        raise
    return {r["arquivo"] for r in linhas}


def ja_ingeridos():
    """Arquivos que ja estao na Bronze."""
    if not spark.catalog.tableExists(TABELA_BRONZE):
        return set()
    linhas = spark.table(TABELA_BRONZE).select("bronze_source_file").distinct().collect()
    return {r["bronze_source_file"] for r in linhas}


no_lake = listar_arquivos()
na_bronze = ja_ingeridos()
print(f"No lake agora  : {len(no_lake)} arquivo(s)")
print(f"Ja na Bronze   : {len(na_bronze)} arquivo(s)")
print(f"A ingerir      : {len(no_lake - na_bronze)} arquivo(s)")

In [0]:
def ingerir(arquivos):
    """Le os arquivos informados e acrescenta na Bronze. Devolve as linhas gravadas."""
    caminhos = [RAIZ_CONTAINER + a for a in sorted(arquivos)]

    ingestao_local = F.expr(f"convert_timezone('{FUSO_NEGOCIO}', bronze_ingested_at)")
    df = (
        spark.read.options(**adls_options)
        .schema(SCHEMA_DT_INTEIRO)
        .parquet(*caminhos)
        .withColumn("bronze_ingested_at", F.current_timestamp())
        .withColumn("bronze_source_file", _relativo(F.col("_metadata.file_path")))
        .withColumn("ano",  F.year(ingestao_local))
        .withColumn("mes",  F.month(ingestao_local))
        .withColumn("dia",  F.dayofmonth(ingestao_local))
        .withColumn("hora", F.hour(ingestao_local))
    )

    (df.write.format("delta")
       .mode("append")
       .partitionBy(*COLUNAS_PARTICAO)
       .saveAsTable(TABELA_BRONZE))

    # Conta pelo que ficou gravado, nao pelo que foi lido: confere a escrita.
    return (spark.table(TABELA_BRONZE)
                 .filter(F.col("bronze_source_file").isin(list(arquivos)))
                 .count())

In [0]:
def agora():
    return datetime.now(ZoneInfo(FUSO_NEGOCIO)).strftime("%H:%M:%S")


ingeridos = ja_ingeridos()
inicio = ultimo_novo = time.monotonic()
ciclos = arquivos_gravados = linhas_gravadas = 0
motivo = "passada unica concluida"

print(f"[{agora()}] inicio - {len(ingeridos)} arquivo(s) ja na Bronze")
try:
    while True:
        ciclos += 1
        novos = listar_arquivos() - ingeridos
        if novos:
            linhas = ingerir(novos)
            ingeridos |= novos
            arquivos_gravados += len(novos)
            linhas_gravadas += linhas
            ultimo_novo = time.monotonic()
            print(f"[{agora()}] +{len(novos)} arquivo(s), {linhas} linha(s)")
            for a in sorted(novos):
                print(f"           {a}")

        if not LOOP_ATIVO:
            break
        decorrido = time.monotonic() - inicio
        if decorrido >= DURACAO_MAX_MIN * 60:
            motivo = f"teto de {DURACAO_MAX_MIN} min atingido"
            break
        if time.monotonic() - ultimo_novo >= OCIOSO_MAX_MIN * 60:
            motivo = f"{OCIOSO_MAX_MIN} min sem arquivo novo - janela provavelmente fechada"
            break
        time.sleep(INTERVALO_S)
except KeyboardInterrupt:
    motivo = "interrompido manualmente"

print(f"[{agora()}] fim: {motivo}")
print(f"Ciclos            : {ciclos}")
print(f"Arquivos gravados : {arquivos_gravados}")
print(f"Linhas gravadas   : {linhas_gravadas}")

In [0]:
if not spark.catalog.tableExists(TABELA_BRONZE):
    print(f"[aviso] {TABELA_BRONZE} ainda nao existe: nenhum arquivo foi ingerido.")
    print("        Rode o loop durante uma janela de tempo real, ou confira se ha arquivos no lake.")
else:
    bronze = spark.table(TABELA_BRONZE)

    por_arquivo = (
        bronze.groupBy("bronze_source_file")
        .agg(F.count("*").alias("linhas"),
             F.countDistinct("bronze_ingested_at").alias("ingestoes"),
             F.min("bronze_ingested_at").alias("ingerido_em"))
        .orderBy("bronze_source_file")
    )
    display(por_arquivo)

    r = bronze.agg(
        F.count("*").alias("linhas"),
        F.countDistinct("bronze_source_file").alias("arquivos"),
        F.countDistinct(CHAVE_PRIMARIA).alias("chaves"),
        F.sum((F.coalesce(F.col("bronze_source_file"), F.lit("")) == "").cast("int")).alias("sem_origem"),
    ).first()
    repetidos = por_arquivo.filter(F.col("ingestoes") > 1).count()
    pendentes = listar_arquivos() - {x["bronze_source_file"] for x in por_arquivo.select("bronze_source_file").collect()}

    print(f"Linhas na Bronze   : {r['linhas']}")
    print(f"Arquivos (lotes)   : {r['arquivos']}")
    print(f"{CHAVE_PRIMARIA} distintos : {r['chaves']}  (repetidos: {r['linhas'] - r['chaves']}, informativo)")
    print()
    checagens = [
        ("arquivos do lake fora da Bronze", len(pendentes)),
        ("arquivos ingeridos mais de uma vez", repetidos),
        ("linhas sem bronze_source_file", r["sem_origem"] or 0),
    ]
    for nome, valor in checagens:
        print(f"[{'ok' if valor == 0 else '!!'}] {nome}: {valor}")
    for a in sorted(pendentes):
        print(f"       pendente: {a}")

    print()
    display(bronze.groupBy(*COLUNAS_PARTICAO).count().orderBy(*COLUNAS_PARTICAO))
    bronze.printSchema()